# Case File: The Broken Reference
A report can still point to a field or measure that no longer exists in its semantic model.
In this investigation, **SFCaseFiles_PNW_Lineage** references a missing measure named **Incident Count**. We will find that reference, replace it with **Total Incidents**, and check the result.
Open the report's **Incident Trend** page first and look at the broken table visual. Our path is: **inspect → repair → verify**.

## 1. Set up the investigation
Use the same setup notebook as the other demos. It installs **Semantic Link Labs 0.18.1** and sets the workspace context.
If setup prompts for a session restart, restart and rerun setup before continuing. The default Lakehouse must be attached, as required by the shared setup notebook.
The example uses a PBIR report named **SFCaseFiles_PNW_Lineage**, connected to the **SFCaseFiles** semantic model. Change the names below to investigate your own report.

In [ ]:
%run 00_Setup_Fresh

In [ ]:
from sempy_labs.semantic_model import lineage_view
from sempy_labs.report import ReportWrapper

DATASET = "SFCaseFiles"
REPORT = "SFCaseFiles_PNW_Lineage"

# See the model's measures and the intended replacement.
measures = fabric.list_measures(dataset=DATASET, workspace=WORKSPACE)
display(measures)

## 2. Find the broken reference
Read the objects used by the report and compare them with the semantic model. With `extended=True`, the result includes **Valid Semantic Model Object**.
Rows marked `False` identify references that no longer resolve. The count is a count of returned references, not necessarily a count of broken visuals.

In [ ]:
# Inspect the objects referenced by this report.
report = ReportWrapper(report=REPORT, workspace=WORKSPACE)
objects_before = report.list_semantic_model_objects(extended=True)

# Find references that no longer match the semantic model.
broken_before = objects_before[
    objects_before["Valid Semantic Model Object"].eq(False)
]

display(broken_before)
print(f"Invalid references before repair: {len(broken_before)}")

## 3. Follow the trail and repair the reference
In the lineage view, select **SFCaseFiles_PNW_Lineage** and analyze its references.
Find the missing **Incident Count** measure, choose **Total Incidents** as its replacement, and save the change. This updates the report.
A replacement must answer the intended business question. An object existing in the model does not, by itself, make it the right replacement.

In [ ]:
lineage_view(dataset=DATASET, workspace=WORKSPACE, dark_mode=True)

## 4. Verify the repair
After saving in the lineage view, read the report definition again and repeat the reference check.
Then reopen or refresh the report's **Incident Trend** page. Confirm that the table visual works and that its values match the intended **Total Incidents** measure. A valid reference is one part of verification; the result must also make sense.

In [ ]:
# Reload the saved report so we inspect the latest definition.
report = ReportWrapper(report=REPORT, workspace=WORKSPACE)
objects_after = report.list_semantic_model_objects(extended=True)

broken_after = objects_after[
    objects_after["Valid Semantic Model Object"].eq(False)
]

display(broken_after)
print(f"Invalid references: {len(broken_before)} before, {len(broken_after)} after")

validity = objects_after["Valid Semantic Model Object"]
if objects_after.empty or validity.isna().any():
    print("Reference check is incomplete. Review the diagnostics before drawing a conclusion.")
elif not validity.isin([True, False]).all():
    print("Reference check returned unexpected values. Review the diagnostics.")
elif broken_after.empty:
    print("No invalid references found. Refresh Incident Trend and verify the visual's values.")
else:
    print("Invalid references remain. Review them in the lineage view.")

## What we learned
We used report metadata to find a missing measure, followed the model's downstream lineage, and repaired the report reference without changing the semantic model.
For another report, change `DATASET` and `REPORT`, inspect its invalid references, and choose replacements based on their business meaning.
### Resources
- [ReportWrapper examples](https://github.com/microsoft/semantic-link-labs/wiki/Code-Examples#find-broken-visuals-in-a-power-bi-report)
- [Semantic Link Labs lineage view](https://semantic-link-labs.readthedocs.io/en/stable/sempy_labs.semantic_model.html#sempy_labs.semantic_model.lineage_view)